# Version 12: Refined Bi-GRU + CRF

**Fixes from V11:**
- **Lower Learning Rate**: Reduced backbone LR from 2e-3 to 8e-4 (stability)
- **Lower Dropout**: Reduced from 0.5 to 0.35 (faster convergence)
- **Sequence Packing**: Proper handling of padding using `pack_padded_sequence`
- **Q8 Only**: Focus capacity on Q8


In [ ]:
!pip install -q pytorch-crf trackio

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from torchcrf import CRF
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Architecture
    embed_dim = 128
    physico_dim = 4
    cnn_channels = 64
    hidden_dim = 384
    num_layers = 2
    dropout = 0.35  # Reduced from 0.5
    
    # Training
    batch_size = 32
    grad_accum = 2
    lr = 8e-4       # Reduced from 2e-3
    weight_decay = 1e-4
    epochs = 60
    grad_clip = 5.0
    use_amp = True
    
    val_split = 0.1
    train_path = 'train.csv'
    test_path = 'test.csv'
    model_save_path = 'best_model_v12.pth'
    submission_path = 'submission_v12.csv'
    
    trackio_project = '25-t3-nppe2'
    trackio_space_id = 'manyakaistha/dlgenai-nppe'
    trackio_group = 'v12_crf_opt'

cfg = Config()
print(f'Device: {cfg.device}')

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

In [ ]:
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_placeholder_token_for_reproducibility'
os.environ['TRACKIO_SPACE_ID'] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        if cfg.trackio_space_id:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            print('✓ TrackIO initialized')
            return trackio
    except Exception as e:
        print(f'TrackIO: {e}')
    return None

trk = maybe_init_trackio()

In [ ]:
AA_PROPS = {
    '<PAD>': [0, 0, 0, 0], '<UNK>': [0, 0, 0, 0], '*': [0, 0, 0, 0],
    'A': [0.4, 0, 0.45, 0.43], 'R': [-1.0, 1, 0.87, 0.77],
    'N': [-0.78, 0, 0.66, 0.39], 'D': [-0.78, -1, 0.67, 0.20],
    'C': [0.56, 0, 0.61, 0.36], 'E': [-0.78, -1, 0.74, 0.23],
    'Q': [-0.78, 0, 0.73, 0.40], 'G': [-0.09, 0, 0.38, 0.43],
    'H': [-0.71, 0.5, 0.78, 0.54], 'I': [1.0, 0, 0.66, 0.43],
    'L': [0.84, 0, 0.66, 0.43], 'K': [-0.87, 1, 0.73, 0.70],
    'M': [0.42, 0, 0.75, 0.41], 'F': [0.62, 0, 0.83, 0.39],
    'P': [-0.36, 0, 0.58, 0.45], 'S': [-0.18, 0, 0.53, 0.41],
    'T': [-0.16, 0, 0.60, 0.40], 'W': [-0.20, 0, 1.02, 0.42],
    'Y': [-0.29, 0, 0.91, 0.40], 'V': [0.93, 0, 0.59, 0.43]
}

PAD_TOKEN = 0
AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

prop_tensor = torch.zeros(len(AA_CHARS), cfg.physico_dim)
for char, idx in AA_TO_ID.items():
    if char in AA_PROPS:
        prop_tensor[idx] = torch.tensor(AA_PROPS[char])

class ProteinDataset(Dataset):
    def __init__(self, df, mode='train'):
        self.df = df
        self.mode = mode
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq, dtype=torch.long), 'id': row['id'], 'len': len(seq)}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID[c] for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID[c] for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    batch.sort(key=lambda x: x['len'], reverse=True)
    seqs = pad_sequence([x['seq'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
    lengths = torch.tensor([x['len'] for x in batch])
    out = {'seq': seqs, 'len': lengths, 'ids': [x['id'] for x in batch]}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([x['q8'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([x['q3'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

df_full = pd.read_csv(cfg.train_path)
train_df, val_df = train_test_split(df_full, test_size=cfg.val_split, random_state=cfg.seed)
train_loader = DataLoader(ProteinDataset(train_df), batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=2)
val_loader = DataLoader(ProteinDataset(val_df), batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=2)

In [ ]:
class BiGRU_CRF(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(AA_TO_ID), cfg.embed_dim, padding_idx=PAD_TOKEN)
        self.prop_embed = nn.Embedding.from_pretrained(prop_tensor, freeze=False)
        self.prop_proj = nn.Linear(cfg.physico_dim, 16)
        
        input_dim = cfg.embed_dim + 16
        self.conv1 = nn.Conv1d(input_dim, cfg.cnn_channels, 3, padding=1)
        self.conv2 = nn.Conv1d(cfg.cnn_channels, cfg.cnn_channels, 5, padding=2)
        self.cnn_norm = nn.LayerNorm(cfg.cnn_channels)
        self.dropout1 = nn.Dropout(cfg.dropout)
        
        self.gru = nn.GRU(cfg.cnn_channels, cfg.hidden_dim, cfg.num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=cfg.dropout if cfg.num_layers > 1 else 0)
        self.gru_norm = nn.LayerNorm(cfg.hidden_dim * 2)
        self.dropout2 = nn.Dropout(cfg.dropout)
        
        self.q8_fc = nn.Linear(cfg.hidden_dim * 2, len(Q8_TO_ID))
        self.q3_fc = nn.Linear(cfg.hidden_dim * 2, len(Q3_TO_ID))
        self.crf_q8 = CRF(len(Q8_TO_ID), batch_first=True)
    
    def forward(self, x, lengths, mask, q8_tags=None, q3_tags=None):
        # Embed
        emb = self.embedding(x)
        props = self.prop_proj(self.prop_embed(x))
        x_in = torch.cat([emb, props], dim=2)
        
        # CNN (requires [B, D, L])
        x_in = x_in.permute(0, 2, 1)
        c1 = torch.relu(self.conv1(x_in))
        c2 = torch.relu(self.conv2(c1))
        x_conv = self.dropout1(self.cnn_norm(c2.permute(0, 2, 1)))
        
        # Packed GRU sequence
        # Move lengths to CPU for pack_padded_sequence
        lengths_cpu = lengths.cpu()
        packed_input = pack_padded_sequence(x_conv, lengths_cpu, batch_first=True, enforce_sorted=True)
        packed_output, _ = self.gru(packed_input)
        gru_out, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=x.size(1))
        
        gru_out = self.dropout2(self.gru_norm(gru_out))
        
        # Heads
        q8_emissions = self.q8_fc(gru_out)
        q3_emissions = self.q3_fc(gru_out)
        
        if q8_tags is not None:
            # Loss
            crf_nll = -self.crf_q8(q8_emissions, q8_tags, mask=mask, reduction='sum')
            loss_q8 = crf_nll / mask.sum()
            loss_q3 = nn.functional.cross_entropy(q3_emissions.view(-1, 9 if len(Q3_TO_ID)>5 else 4), 
                                                 q3_tags.view(-1), ignore_index=PAD_TOKEN)
            return loss_q8, loss_q3
        else:
            return self.crf_q8.decode(q8_emissions, mask=mask), q3_emissions.argmax(-1)

model = BiGRU_CRF().to(cfg.device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = GradScaler(enabled=cfg.use_amp)

def harmonic_mean(a, b): return 2 * (a * b) / (a + b + 1e-8)

def train_step(loader, epoch):
    model.train()
    total_loss, total_q8, total_q3 = 0, 0, 0
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f'Ep {epoch+1}')
    for step, batch in enumerate(pbar):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['len'].to(cfg.device)
        mask = (seq != PAD_TOKEN)
        q8 = batch['q8'].to(cfg.device)
        q3 = batch['q3'].to(cfg.device)
        
        with autocast(device_type='cuda', enabled=cfg.use_amp):
            l8, l3 = model(seq, lengths, mask, q8, q3)
            loss = (l8 + l3) / cfg.grad_accum
        
        scaler.scale(loss).backward()
        
        if (step+1) % cfg.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step(epoch + step/len(loader))
        
        total_loss += (l8.item() + l3.item())
        pbar.set_postfix({'loss': total_loss/(step+1)})
        
        if trk and step % 20 == 0: trk.log({'loss': l8.item()+l3.item()})
    return total_loss/len(loader)

def validate(loader):
    model.eval()
    all_q8_p, all_q8_t, all_q3_p, all_q3_t = [], [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Val'):
            seq = batch['seq'].to(cfg.device)
            lengths = batch['len'].to(cfg.device)
            mask = (seq != PAD_TOKEN)
            
            with autocast(device_type='cuda', enabled=cfg.use_amp):
                p8_list, p3 = model(seq, lengths, mask)
            
            for i, L in enumerate(batch['len']):
                all_q8_p.extend(p8_list[i][:L])
                all_q8_t.extend(batch['q8'][i, :L].numpy())
                all_q3_p.extend(p3[i, :L].cpu().numpy())
                all_q3_t.extend(batch['q3'][i, :L].numpy())
                
    f1_8 = f1_score(all_q8_t, all_q8_p, average='macro', zero_division=0)
    f1_3 = f1_score(all_q3_t, all_q3_p, average='macro', zero_division=0)
    return f1_8, f1_3, harmonic_mean(f1_8, f1_3)

best = 0
for ep in range(cfg.epochs):
    loss = train_step(train_loader, ep)
    q8, q3, harm = validate(val_loader)
    print(f'Ep {ep+1}: Loss={loss:.4f} | Q8={q8:.4f} Q3={q3:.4f} Harm={harm:.4f}')
    if harm > best:
        best = harm
        torch.save(model.state_dict(), cfg.model_save_path)
        print('✓ Saved New Best!')
    if trk:
        trk.log({'val_harm': harm, 'val_q8': q8})

In [ ]:
model.load_state_dict(torch.load(cfg.model_save_path))
model.eval()
test_df = pd.read_csv(cfg.test_path)
test_loader = DataLoader(ProteinDataset(test_df, 'test'), batch_size=cfg.batch_size, collate_fn=collate_fn)
res = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['len'].to(cfg.device)
        mask = (seq != PAD_TOKEN)
        p8, p3 = model(seq, lengths, mask)
        for i, L in enumerate(batch['len']):
            s8 = ''.join([ID_TO_Q8[x] for x in p8[i][:L]])
            s3 = ''.join([ID_TO_Q3[x] for x in p3[i, :L].cpu().numpy()])
            res.append({'id': batch['ids'][i], 'sst8': s8, 'sst3': s3})
pd.DataFrame(res).to_csv(cfg.submission_path, index=False)
print('✓ Done')